In [17]:

import numpy as np
import matplotlib.pyplot as plt
from helpers import *
import importlib, helpers
importlib.reload(helpers)
from helpers import standardize, build_model_data

In [18]:
x_train, x_test, y_train, train_ids, test_ids = load_csv_data("..\dataset")

In [3]:
y_train.shape, x_train.shape, x_train.shape

((328135,), (328135, 321), (328135, 321))

In [20]:
def load_feature_names(data_path):
    with open(os.path.join(data_path, "x_train.csv"), "r") as f:
        header = f.readline().strip().split(",")  # read first line
    return header[1:]  # skip the "Id" column

In [21]:
feature_names = load_feature_names("..\dataset")
#print(feature_names)
feature_to_index = {name: i for i, name in enumerate(feature_names)}
print(feature_to_index)

{'_STATE': 0, 'FMONTH': 1, 'IDATE': 2, 'IMONTH': 3, 'IDAY': 4, 'IYEAR': 5, 'DISPCODE': 6, 'SEQNO': 7, '_PSU': 8, 'CTELENUM': 9, 'PVTRESD1': 10, 'COLGHOUS': 11, 'STATERES': 12, 'CELLFON3': 13, 'LADULT': 14, 'NUMADULT': 15, 'NUMMEN': 16, 'NUMWOMEN': 17, 'CTELNUM1': 18, 'CELLFON2': 19, 'CADULT': 20, 'PVTRESD2': 21, 'CCLGHOUS': 22, 'CSTATE': 23, 'LANDLINE': 24, 'HHADULT': 25, 'GENHLTH': 26, 'PHYSHLTH': 27, 'MENTHLTH': 28, 'POORHLTH': 29, 'HLTHPLN1': 30, 'PERSDOC2': 31, 'MEDCOST': 32, 'CHECKUP1': 33, 'BPHIGH4': 34, 'BPMEDS': 35, 'BLOODCHO': 36, 'CHOLCHK': 37, 'TOLDHI2': 38, 'CVDSTRK3': 39, 'ASTHMA3': 40, 'ASTHNOW': 41, 'CHCSCNCR': 42, 'CHCOCNCR': 43, 'CHCCOPD1': 44, 'HAVARTH3': 45, 'ADDEPEV2': 46, 'CHCKIDNY': 47, 'DIABETE3': 48, 'DIABAGE2': 49, 'SEX': 50, 'MARITAL': 51, 'EDUCA': 52, 'RENTHOM1': 53, 'NUMHHOL2': 54, 'NUMPHON2': 55, 'CPDEMO1': 56, 'VETERAN3': 57, 'EMPLOY1': 58, 'CHILDREN': 59, 'INCOME2': 60, 'INTERNET': 61, 'WEIGHT2': 62, 'HEIGHT3': 63, 'PREGNANT': 64, 'QLACTLM2': 65, 'USEEQUI

In [5]:
# keeping just the features we want
features_to_keep = [26, 30, 32, 34, 35, 38, 39, 40, 43, 44, 45, 46, 47, 48, 50,52, 57, 58, 60,62, 63, 72, 73, 75, 76, 77, 82, 83, 84, 85, 86, 87, 89, 104, 136, 142, 144, 155, 156, 192, 193, 234, 235, 238, 242, 248, 250, 251, 252, 253, 259, 264, 276, 277, 301]
# those not to clean from outliers
x_train_ntc = x_train.copy()
x_train_ntc = x_train[:, [235,242,46,44,47,43,136,39,48,30,32,76,57]]
features_to_keep = [26, 30, 32, 34, 35, 38, 39, 40, 43, 44, 45, 46, 47, 48, 50,
                    52, 57, 58, 60, 62, 63, 72, 73, 75, 76, 77, 82, 83, 84, 85,
                    86, 87, 89, 104, 136, 142, 144, 155, 156, 192, 193, 234, 235,
                    238, 242, 248, 250, 251, 252, 253, 259, 264, 276, 277, 301]

features_w_outliers = [235, 242, 46, 44, 47, 43, 136, 39, 48, 30, 32, 76, 57]

# Convert to numpy arrays
to_keep = np.array(features_to_keep)
with_outliers = np.array(features_w_outliers)

# Find elements in features_to_keep that are NOT in features_w_outliers
diff_indices = np.setdiff1d(to_keep, with_outliers)

x_train_toclean = x_train.copy()
x_train_toclean[:, diff_indices] = x_train[:, diff_indices]
x_train = x_train[:, features_to_keep]
x_test = x_test[:, features_to_keep]
print(x_train.shape)

(328135, 55)


In [19]:
features_w_outliers = [235,242,46,44,47,43,136,39,48,30,32,76,57]
features_to_keep = [26, 30, 32, 34, 35, 38, 39, 40, 43, 44, 45, 46, 47, 48, 50,
                    52, 57, 58, 60, 62, 63, 72, 73, 75, 76, 77, 82, 83, 84, 85,
                    86, 87, 89, 104, 136, 142, 144, 155, 156, 192, 193, 234, 235,
                    238, 242, 248, 250, 251, 252, 253, 259, 264, 276, 277, 301]

features_w_outliers = [235, 242, 46, 44, 47, 43, 136, 39, 48, 30, 32, 76, 57]


In [10]:
# boxplots w/ outliers
import os

# Create output folder for boxplots
boxplot_dir = "../boxplots_woutliers"
os.makedirs(boxplot_dir, exist_ok=True)

for i, feature_idx in enumerate(range(x_train_ntc.shape[1])):
    feature_name = feature_names[features_w_outliers[i]]
    data = x_train_ntc[:, feature_idx]
    # General boxplot
    data_general = data[~np.isnan(data)]
    # Boxplot for y_train == -1
    data_neg1 = data[(y_train == -1) & (~np.isnan(data))]
    # Boxplot for y_train == 1
    data_pos1 = data[(y_train == 1) & (~np.isnan(data))]

    plt.figure(figsize=(8, 6))
    plt.boxplot([data_general, data_neg1, data_pos1], tick_labels=["All", "y=-1", "y=1"], showfliers=True)
    plt.title(feature_name)
    plt.ylabel("Value")
    plt.tight_layout()
    plt.savefig(os.path.join(boxplot_dir, f"{feature_name}.png"))
    plt.close()

In [ ]:
# Replace outliers in x_train with NaNs using the IQR method
Q1 = np.nanpercentile(x_train, 25, axis=0)
Q3 = np.nanpercentile(x_train, 75, axis=0)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = (x_train < lower_bound) | (x_train > upper_bound)
x_train_clean = x_train.copy()
x_train_clean[outliers] = np.nan

In [ ]:
import os

# Create output folder for boxplots
boxplot_dir = "../boxplots"
os.makedirs(boxplot_dir, exist_ok=True)

for i, feature_idx in enumerate(range(x_train_clean.shape[1])):
    feature_name = feature_names[features_to_keep[i]]
    data = x_train_clean[:, feature_idx]
    # General boxplot
    data_general = data[~np.isnan(data)]
    # Boxplot for y_train == -1
    data_neg1 = data[(y_train == -1) & (~np.isnan(data))]
    # Boxplot for y_train == 1
    data_pos1 = data[(y_train == 1) & (~np.isnan(data))]

    plt.figure(figsize=(8, 6))
    plt.boxplot([data_general, data_neg1, data_pos1], tick_labels=["All", "y=-1", "y=1"], showfliers=False)
    plt.title(feature_name)
    plt.ylabel("Value")
    plt.tight_layout()
    plt.savefig(os.path.join(boxplot_dir, f"{feature_name}.png"))
    plt.close()

In [13]:
# Divide features in x_train_clean into continuous and categorical

continuous_features = []
categorical_features = []

for idx, feature_idx in enumerate(features_to_keep):
    name = feature_names[feature_idx]
    col = x_train_clean[:, idx]
    unique_vals = np.unique(col[~np.isnan(col)])
    # Heuristic: categorical if <= 10 unique values and all are integers
    if len(unique_vals) <= 10 and np.all(np.equal(np.mod(unique_vals, 1), 0)):
        categorical_features.append(name)
    else:
        continuous_features.append(name)

print("Continuous features:", continuous_features)
print("Categorical features:", categorical_features)

Continuous features: ['WEIGHT2', 'HEIGHT3', 'ALCDAY5', 'FRUIT1', 'FVBEANS', 'FVGREEN', 'FVORANG', 'VEGETAB1', 'EXEROFT1', '_AGE80', 'HTIN4', 'HTM4', 'WTKG3', '_BMI5', '_DRNKWEK', '_FRUTSUM', '_VEGESUM', 'PA1MIN_']
Categorical features: ['GENHLTH', 'HLTHPLN1', 'MEDCOST', 'BPHIGH4', 'BPMEDS', 'TOLDHI2', 'CVDSTRK3', 'ASTHMA3', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY', 'DIABETE3', 'SEX', 'EDUCA', 'VETERAN3', 'EMPLOY1', 'INCOME2', 'SMOKE100', 'SMOKDAY2', 'LASTSMK2', 'USENOW3', 'EXERANY2', 'HIVTST6', 'CIMEMLOS', 'WTCHSALT', 'DRADVISE', 'HAREHAB1', 'STREHAB1', 'SCNTMNY1', 'SCNTMEL1', '_RFCHOL', '_LTASTH1', '_DRDXAR1', '_RACE', '_SMOKER3']


In [ ]:
import pandas as pd

# Compute statistics for each feature in x_train_clean
stats = []
for i in range(x_train_clean.shape[1]):
    col = x_train_clean[:, i]
    col_nonan = col[~np.isnan(col)]
    stats.append({
        "Feature": feature_names[features_to_keep[i]],
        "Mean": np.mean(col_nonan),
        "Median": np.median(col_nonan),
        "Variance": np.var(col_nonan),
        "Min": np.percentile(col_nonan, 0),
        "Q1": np.percentile(col_nonan, 25),
        "Q2": np.percentile(col_nonan, 50),
        "Q3": np.percentile(col_nonan, 75),
        "Max": np.percentile(col_nonan, 100)
    })

stats_df = pd.DataFrame(stats)
stats_df.to_csv("feature_statistics.csv", index=False)

# Save as image
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, len(stats_df) * 0.3))
ax.axis('off')
table = ax.table(cellText=stats_df.values, colLabels=stats_df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.auto_set_column_width(col=list(range(len(stats_df.columns))))
plt.tight_layout()
plt.savefig("feature_statistics.png")
plt.close()

In [ ]:
import pandas as pd

# Compute statistics for each feature in x_train_clean correpsonding to y_train values positive
stats = []
for i in range(x_train_clean.shape[1]):
    col = x_train_clean[:, i]
    col_nonan = col[(~np.isnan(col)) & (y_train == 1)]
    stats.append({
        "Feature": feature_names[features_to_keep[i]],
        "Mean": np.mean(col_nonan),
        "Median": np.median(col_nonan),
        "Variance": np.var(col_nonan),
        "Min": np.percentile(col_nonan, 0),
        "Q1": np.percentile(col_nonan, 25),
        "Q2": np.percentile(col_nonan, 50),
        "Q3": np.percentile(col_nonan, 75),
        "Max": np.percentile(col_nonan, 100)
    })

stats_df = pd.DataFrame(stats)
stats_df.to_csv("feature_statistics_YMICHD.csv", index=False)

# Save as image
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, len(stats_df) * 0.3))
ax.axis('off')
table = ax.table(cellText=stats_df.values, colLabels=stats_df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.auto_set_column_width(col=list(range(len(stats_df.columns))))
plt.tight_layout()
plt.savefig("feature_statistics_YMICHD.png")
plt.close()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

# Collect statistics
stats = []
header = ["Feature", "Mean", "Median", "Variance", "Min", "Q1", "Q2", "Q3", "Max"]

for i in range(x_train_clean.shape[1]):
    col = x_train_clean[:, i]
    col_nonan = col[(~np.isnan(col)) & (y_train == -1)]

    if col_nonan.size == 0:
        stats.append([
            feature_names[features_to_keep[i]],
            np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
        ])
        continue

    stats.append([
        feature_names[features_to_keep[i]],
        np.mean(col_nonan),
        np.median(col_nonan),
        np.var(col_nonan),
        np.min(col_nonan),
        np.percentile(col_nonan, 25),
        np.percentile(col_nonan, 50),
        np.percentile(col_nonan, 75),
        np.max(col_nonan)
    ])

# --- Save as CSV manually ---
csv_filename = "feature_statistics_NMICHD.csv"
with open(csv_filename, "w", newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(stats)

print(f"✅ CSV saved as {csv_filename}")

# --- Save as image (using Matplotlib Table) ---
fig, ax = plt.subplots(figsize=(12, len(stats) * 0.3))
ax.axis('off')
table = ax.table(cellText=stats, colLabels=header, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.auto_set_column_width(col=list(range(len(header))))
plt.tight_layout()
plt.savefig("feature_statistics_NMICHD.png")
plt.close()

print("✅ Image saved as feature_statistics_NMICHD.png")


In [ ]:
import os

import matplotlib.pyplot as plt

# Create output folder for histograms of relative frequency
hist_relfreq_dir = "hists_relfreq"
os.makedirs(hist_relfreq_dir, exist_ok=True)

for i in range(x_train_clean.shape[1]):
    feature_name = feature_names[features_to_keep[i]]
    col = x_train_clean[:, i]
    mask_neg1 = (y_train == -1) & (~np.isnan(col))
    mask_pos1 = (y_train == 1) & (~np.isnan(col))
    data_neg1 = col[mask_neg1]
    data_pos1 = col[mask_pos1]

    # Use bins based on unique values for categorical, or 30 bins for continuous
    if feature_name in categorical_features:
        bins = np.unique(np.concatenate([data_neg1, data_pos1]))
    else:
        bins = 30

    plt.figure(figsize=(8, 6))
    plt.hist(data_neg1, bins=bins, density=True, alpha=0.6, color='blue', label='y = -1', edgecolor='black')
    plt.hist(data_pos1, bins=bins, density=True, alpha=0.6, color='red', label='y = 1', edgecolor='black')
    plt.title(f"Relative Frequency Histogram of {feature_name} by y_train")
    plt.xlabel(feature_name)
    plt.ylabel("Relative Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(hist_relfreq_dir, f"{feature_name}_relfreq.png"))
    plt.close()

In [19]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Create output folder
hist_relfreq_dir = "../hists_relfreq_woutliers"
os.makedirs(hist_relfreq_dir, exist_ok=True)

for i in range(x_train_ntc.shape[1]):
    feature_name = feature_names[features_w_outliers[i]]
    col = x_train_ntc[:, i]

    mask_neg1 = (y_train == -1) & (~np.isnan(col))
    mask_pos1 = (y_train == 1) & (~np.isnan(col))
    data_neg1 = col[mask_neg1]
    data_pos1 = col[mask_pos1]

    plt.figure(figsize=(8, 6))

    if feature_name in categorical_features:
        unique_vals = np.unique(np.concatenate([data_neg1, data_pos1]))

        # Plot each category separately to allow gaps
        for val in unique_vals:
            plt.bar(val, np.sum(data_neg1 == val) / len(data_neg1), 
                    width=1, color='blue', alpha=0.6, edgecolor='black', label='y=-1' if val == unique_vals[0] else "")
            plt.bar(val, np.sum(data_pos1 == val) / len(data_pos1), 
                    width=1, color='red', alpha=0.6, edgecolor='black', label='y=1' if val == unique_vals[0] else "")

        plt.xticks(unique_vals)
    else:
        # Continuous features
        plt.hist(data_neg1, bins=30, density=True, alpha=0.6, color='blue', edgecolor='black', label='y=-1')
        plt.hist(data_pos1, bins=30, density=True, alpha=0.6, color='red', edgecolor='black', label='y=1')

    plt.title(f"Relative Frequency Histogram of {feature_name} by y_train")
    plt.xlabel(feature_name)
    plt.ylabel("Relative Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(hist_relfreq_dir, f"{feature_name}_relfreq.png"))
    plt.close()


In [ ]:
import os

import matplotlib.pyplot as plt

# Create output folder for histograms of frequency
hist_dir = "hists"
os.makedirs(hist_dir, exist_ok=True)

for i in range(x_train_clean.shape[1]):
    feature_name = feature_names[features_to_keep[i]]
    col = x_train_clean[:, i]
    mask_neg1 = (y_train == -1) & (~np.isnan(col))
    mask_pos1 = (y_train == 1) & (~np.isnan(col))
    data_neg1 = col[mask_neg1]
    data_pos1 = col[mask_pos1]

    # Use bins based on unique values for categorical, or 30 bins for continuous
    if feature_name in categorical_features:
        bins = np.unique(np.concatenate([data_neg1, data_pos1]))
    else:
        bins = 30

    plt.figure(figsize=(8, 6))
    plt.hist(data_neg1, bins=bins, density=False, alpha=0.6, color='blue', label='y = -1', edgecolor='black')
    plt.hist(data_pos1, bins=bins, density=False, alpha=0.6, color='red', label='y = 1', edgecolor='black')
    plt.title(f"Histogram of {feature_name} by y_train")
    plt.xlabel(feature_name)
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(hist_dir, f"{feature_name}_hist.png"))
    plt.close()

In [ ]:
import os

import matplotlib.pyplot as plt

# Compute correlation matrix for x_train_clean (pairwise Pearson, ignoring NaNs)
corr_matrix = np.full((x_train_clean.shape[1], x_train_clean.shape[1]), np.nan)
for i in range(x_train_clean.shape[1]):
    for j in range(x_train_clean.shape[1]):
        col_i = x_train_clean[:, i]
        col_j = x_train_clean[:, j]
        mask = ~np.isnan(col_i) & ~np.isnan(col_j)
        if np.sum(mask) > 1 :
            std_i = np.std(col_i[mask])
            std_j = np.std(col_j[mask])
            if std_i == 0 or std_j == 0:
                corr_matrix[i,j] = np.nan
            else:
                corr_matrix[i,j] = np.corrcoef(col_i[mask], col_j[mask])[0,1]
np.fill_diagonal(corr_matrix, 1.0)

# Mark non-negligible correlations (absolute value > 0.3)
marked_005 = np.where(np.abs(corr_matrix) > 0.05, corr_matrix, np.nan)

# Plot and save
heatmap_dir = "heatmaps"
os.makedirs(heatmap_dir, exist_ok=True)

# Plot and save for |corr| > 0.05
plt.figure(figsize=(14, 12))
im = plt.imshow(marked_005, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.title("Correlation Matrix (|corr| > 0.05 marked)")
plt.xticks(range(len(features_to_keep)), [feature_names[i] for i in features_to_keep], rotation=90, fontsize=8)
plt.yticks(range(len(features_to_keep)), [feature_names[i] for i in features_to_keep], fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(heatmap_dir, "correlation_matrix_marked_005.png"))
plt.close()

# Subgroups: continuous and categorical features

# Other subgroups: define custom ranges of feature indices (by column in x_train_clean)
subgroups = [
    (0, 4),
    (2, 6),
    (4, 8),
    (6, 10),
    (8, 12),
    (10, 14),
    (12, 16),
    (14, 18),
    (16, 20),
    (18, 22),
    (20, 24),
    (22, 26),
    (24, 28),
    (26, 30),
    (28, 32),
    (30, 34),
    (32, 36),
    (34, 38),
    (36, 40),
    (38, 42),
    (40, 44),
    (42, 46),
    (44, 48),
    (46, 50),
    (48, 52),
    (50, 54)
]

for start, end in subgroups:
    subgroup_names = [feature_names[features_to_keep[i]] for i in range(start, end + 1)]
    corr_sub = corr_matrix[start:end+1, start:end+1]
    marked_sub = np.where(np.abs(corr_sub) > 0.05, corr_sub, np.nan)
    plt.figure(figsize=(6, 5))
    im = plt.imshow(marked_sub, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.title(f"Correlation Matrix (Features {start+1} to {end+1}, |corr| > 0.05 marked)")
    plt.xticks(range(len(subgroup_names)), subgroup_names, rotation=90, fontsize=8)
    plt.yticks(range(len(subgroup_names)), subgroup_names, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(heatmap_dir, f"correlation_matrix_subgroup_{start+1}_{end+1}.png"))
    plt.close()
continuous_idxs = [features_to_keep.index(feature_to_index[name]) for name in continuous_features if name in feature_to_index and feature_to_index[name] in features_to_keep]
categorical_idxs = [features_to_keep.index(feature_to_index[name]) for name in categorical_features if name in feature_to_index and feature_to_index[name] in features_to_keep]

# Continuous features correlation matrix
if continuous_idxs:
    corr_cont = corr_matrix[np.ix_(continuous_idxs, continuous_idxs)]
    marked_cont = np.where(np.abs(corr_cont) > 0.05, corr_cont, np.nan)
    plt.figure(figsize=(10, 8))
    im = plt.imshow(marked_cont, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.title("Correlation Matrix (Continuous Features, |corr| > 0.05 marked)")
    plt.xticks(range(len(continuous_idxs)), [continuous_features[i] for i in range(len(continuous_idxs))], rotation=90, fontsize=8)
    plt.yticks(range(len(continuous_idxs)), [continuous_features[i] for i in range(len(continuous_idxs))], fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(heatmap_dir, "correlation_matrix_continuous.png"))
    plt.close()

# Categorical features correlation matrix
if categorical_idxs:
    corr_cat = corr_matrix[np.ix_(categorical_idxs, categorical_idxs)]
    marked_cat = np.where(np.abs(corr_cat) > 0.05, corr_cat, np.nan)
    plt.figure(figsize=(10, 8))
    im = plt.imshow(marked_cat, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.title("Correlation Matrix (Categorical Features, |corr| > 0.05 marked)")
    plt.xticks(range(len(categorical_idxs)), [categorical_features[i] for i in range(len(categorical_idxs))], rotation=90, fontsize=8)
    plt.yticks(range(len(categorical_idxs)), [categorical_features[i] for i in range(len(categorical_idxs))], fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(heatmap_dir, "correlation_matrix_categorical.png"))
    plt.close()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

# Prepare header
header = [
    "Feature",
    "NaN count (y=-1)", "NaN % (y=-1)",
    "NaN count (y=1)", "NaN % (y=1)",
    "NaN count (All)", "NaN % (All)"
]

stats_nan = []

# Precompute masks
mask_neg = (y_train == -1)
mask_pos = (y_train == 1)

n_neg = np.sum(mask_neg)
n_pos = np.sum(mask_pos)
n_all = len(y_train)

for i in range(x_train_clean.shape[1]):
    col = x_train_clean[:, i]

    # Count NaNs
    nan_neg = np.sum(np.isnan(col[mask_neg]))
    nan_pos = np.sum(np.isnan(col[mask_pos]))
    nan_all = np.sum(np.isnan(col))

    # Percentages (handle division by zero safely)
    pct_neg = (nan_neg / n_neg * 100) if n_neg > 0 else np.nan
    pct_pos = (nan_pos / n_pos * 100) if n_pos > 0 else np.nan
    pct_all = (nan_all / n_all * 100) if n_all > 0 else np.nan

    stats_nan.append([
        feature_names[features_to_keep[i]],
        int(nan_neg), round(pct_neg, 2),
        int(nan_pos), round(pct_pos, 2),
        int(nan_all), round(pct_all, 2)
    ])

# --- Save as CSV ---
csv_filename = "feature_nan_counts_and_percentages.csv"
with open(csv_filename, "w", newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(stats_nan)

# --- Save as image (table) ---
fig_height = max(3, len(stats_nan) * 0.3)
fig, ax = plt.subplots(figsize=(10, fig_height))
ax.axis('off')
table = ax.table(cellText=stats_nan, colLabels=header, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.auto_set_column_width(col=list(range(len(header))))
plt.tight_layout()
plt.savefig("feature_nan_counts_and_percentages.png")
plt.close()




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Select continuous feature indices from features_to_keep
continuous_idxs = [features_to_keep.index(feature_to_index[name])
                   for name in continuous_features
                   if name in feature_to_index and feature_to_index[name] in features_to_keep]

# Extract data for continuous features
X_cont = x_train_clean[:, continuous_idxs]
feature_labels = [feature_names[features_to_keep[i]] for i in continuous_idxs]

# Colors and markers by class
colors = {-1: "tab:blue", 1: "tab:orange"}
markers = {-1: "o", 1: "^"}

# Number of features
n = len(feature_labels)
if n < 2:
    raise ValueError("Need at least 2 continuous features to make scatterplots.")

# Create grid of subplots
fig, axes = plt.subplots(n, n, figsize=(2.5 * n, 2.5 * n))
fig.suptitle("Scatterplot Matrix — Continuous Features (masked by y_train)", fontsize=14)

for i in range(n):
    for j in range(n):
        ax = axes[i, j]

        if i == j:
            # Histogram of feature on diagonal
            data = X_cont[:, i]
            mask = ~np.isnan(data)
            ax.hist(data[mask], bins=20, color="gray", alpha=0.7)
            ax.set_ylabel(feature_labels[i])
            ax.set_xlabel(feature_labels[j])
        else:
            # Scatterplots masked by y_train
            xi = X_cont[:, j]
            yi = X_cont[:, i]
            mask_notnan = ~np.isnan(xi) & ~np.isnan(yi)

            for y_val in [-1, 1]:
                mask_class = (y_train == y_val) & mask_notnan
                ax.scatter(
                    xi[mask_class],
                    yi[mask_class],
                    s=10,
                    alpha=0.6,
                    c=colors[y_val],
                    marker=markers[y_val],
                    label=f"y={y_val}" if i == 0 and j == 0 else None
                )

        # Axis labeling
        if i == n - 1:
            ax.set_xlabel(feature_labels[j], rotation=45, fontsize=8)
        else:
            ax.set_xticklabels([])
        if j == 0:
            ax.set_ylabel(feature_labels[i], fontsize=8)
        else:
            ax.set_yticklabels([])

# Add legend once
axes[0, 0].legend(fontsize=8)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()  # You can replace with plt.savefig("scatter_continuous.png")




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Get indices of continuous features (same logic as before)
continuous_idxs = [features_to_keep.index(feature_to_index[name])
                   for name in continuous_features
                   if name in feature_to_index and feature_to_index[name] in features_to_keep]

# Extract data and names
X_cont = x_train_clean[:, continuous_idxs]
feature_labels = [feature_names[features_to_keep[i]] for i in continuous_idxs]

# Colors and markers for y_train classes
colors = {-1: "tab:blue", 1: "tab:orange"}
markers = {-1: "o", 1: "^"}

# Define subgroups of 5 features, stepping by 2
subgroups = []
for start in range(0, len(feature_labels) - 1, 2):
    end = min(start + 4, len(feature_labels) - 1)
    subgroups.append((start, end))

print(f"Creating {len(subgroups)} subgroups...")

for group_idx, (start, end) in enumerate(subgroups, start=1):
    # Extract subgroup
    X_sub = X_cont[:, start:end + 1]
    sub_labels = feature_labels[start:end + 1]
    n = X_sub.shape[1]

    if n < 2:
        continue  # Need at least 2 to make scatterplots

    fig, axes = plt.subplots(n, n, figsize=(2.5 * n, 2.5 * n))
    fig.suptitle(f"Continuous Features {start+1}–{end+1}", fontsize=14)

    for i in range(n):
        for j in range(n):
            ax = axes[i, j]

            if i == j:
                # Histogram on diagonal
                data = X_sub[:, i]
                mask = ~np.isnan(data)
                ax.hist(data[mask], bins=20, color="gray", alpha=0.7)
            else:
                # Scatterplot masked by y_train
                xi = X_sub[:, j]
                yi = X_sub[:, i]
                mask_notnan = ~np.isnan(xi) & ~np.isnan(yi)
                for y_val in [-1, 1]:
                    mask_class = (y_train == y_val) & mask_notnan
                    ax.scatter(
                        xi[mask_class],
                        yi[mask_class],
                        s=10,
                        alpha=0.6,
                        c=colors[y_val],
                        marker=markers[y_val],
                    )

            # Label formatting
            if i == n - 1:
                ax.set_xlabel(sub_labels[j], rotation=45, fontsize=8)
            else:
                ax.set_xticklabels([])
            if j == 0:
                ax.set_ylabel(sub_labels[i], fontsize=8)
            else:
                ax.set_yticklabels([])

    # Add small legend on first plot
    axes[0, 0].scatter([], [], c=colors[-1], marker=markers[-1], label="y=-1", s=20)
    axes[0, 0].scatter([], [], c=colors[1], marker=markers[1], label="y=1", s=20)
    axes[0, 0].legend(fontsize=8)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(f"scatter_continuous_group_{start+1}_{end+1}.png", dpi=200)
    plt.close(fig)

print("✅ Scatterplots saved for all subgroups.")
